# Grupimi dhe zbulimi i strukturës

**Lënda:** Machine Learning in Physics  
**Mjedisi:** Python, NumPy, pandas, Matplotlib dhe scikit-learn  
**Roli i notebook-ut:** mjeti kryesor i punës gjatë orës; leksioni i shkruar shërben për thellim teorik.

Në grupimin e pambikëqyrur nuk kemi etiketa që na tregojnë përgjigjen. Kjo e ndryshon natyrën e pyetjes: algoritmi nuk “zbulon të vërtetën”, por propozon një strukturë sipas një kriteri të caktuar gjeometrik. Ky laborator e bën këtë kufizim të dukshëm duke ndërtuar k-means hap pas hapi dhe duke treguar raste ku ai funksionon mirë dhe raste ku dështon.

Për këtë temë do t'u përgjigjemi sidomos tri pyetjeve:

1. Çfarë funksioni objektiv minimizon k-means?
2. Si ndikon inicializimi?
3. Si vendosim nëse numri i grupeve është i arsyeshëm?

Ritmi i punës është: **parashiko → formulo matematikisht → implemento → ekzekuto → mat → interpreto → provo ta prishësh modelin**. Qelizat nuk duhen ekzekutuar mekanikisht njëra pas tjetrës. Para çdo eksperimenti duhet të kemi një pritshmëri dhe pas tij duhet të shpjegojmë mekanizmin që prodhoi rezultatin.


## Organizimi i seancës

| Koha | Veprimtaria | Rezultati i pritshëm |
|---:|---|---|
| 0–10 min | Pyetja shkencore dhe formulimi matematikor | Kuptimi i objektit që po modelojmë |
| 10–25 min | Zbatimi nga parimet e para | Lidhja ndërmjet formulës dhe algoritmit |
| 25–40 min | Eksperimenti numerik dhe figurat diagnostike | Kontrolli i sjelljes së modelit |
| 40–50 min | Krahasimi me implementimin profesional | Verifikimi i zbatimit dhe praktikës së mirë |
| 50–60 min | Interpretimi, kufizimet dhe një rast dështimi | Përfundim metodologjik, jo vetëm numerik |
| Pas orës | Punishtja dhe ushtrimet | Ndrysho një parametër, mat efektin dhe shpjego mekanizmin |

Kjo tabelë është orientuese. Nëse një diskutim fizik ose matematik është i frytshëm, është më e vlefshme të ndalemi te ai sesa të “mbarojmë” notebook-un.


## Rezultatet e të nxënit

Në fund të punës studenti duhet të jetë në gjendje të:

- Shpjegoj hapat assignment/update.
- Interpretoj inertia.
- Përdor silhouette me kujdes.
- Lexoj një dendrogram.
- Tregoj një rast ku k-means jep ndarje mashtruese.


### Para se të ekzekutojmë: Mjedisi llogaritës dhe riprodhueshmëria

Ngarkojmë bibliotekat që do të përdoren dhe fiksojmë `RANDOM_STATE = 42`. Ky seed nuk e bën problemin determinist në kuptimin fizik; ai thjesht na lejon të përsërisim të njëjtin eksperiment llogaritës dhe të diskutojmë metodën pa u hutuar nga ndryshime të rastësishme të mostrës.

**Parashikim para ekzekutimit.** Vlerësoni rendin e madhësisë dhe drejtimin e rezultatit përpara se ta llogarisë kompjuteri.


In [ ]:
# Qëllimi i këtij blloku: Mjedisi llogaritës dhe riprodhueshmëria.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import numpy as np
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import pandas as pd
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import matplotlib.pyplot as plt
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from pathlib import Path
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import time

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

print("09 — Grupimi dhe zbulimi i strukturës")
print("Seed:", RANDOM_STATE)

**Interpretim.** Një dalje numerike merr kuptim vetëm pasi krahasohet me shkallën e problemit, pritshmërinë teorike dhe një kontroll të pavarur kur është e mundur.


# 1. Regjime atmosferike pa etiketa — CORE

### Para se të ekzekutojmë: Të dhëna pa etiketa

Heqim etiketat dhe ruajmë vetëm koordinatat. Kjo ndryshon natyrën e problemit: nuk kemi më një përgjigje të njohur ndaj së cilës optimizojmë accuracy; duhet të gjykojmë strukturën përmes gjeometrisë dhe interpretimit.

**Parashikim para ekzekutimit.** Vendosni çfarë forme duhet të ketë figura nëse argumenti teorik është i saktë. Grafiku përdoret për të testuar një pritshmëri, jo vetëm për ta ilustruar.


In [ ]:
# Qëllimi i këtij blloku: Të dhëna pa etiketa.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.datasets import make_blobs
# Gjenerojmë të dhëna të kontrolluara që na lejojnë të dimë strukturën që duhet të zbulojë modeli.
X,_=make_blobs(n_samples=520,centers=[[-2,-1],[0,2.3],[2.6,-0.4]],
               cluster_std=[0.6,0.75,0.65],random_state=RANDOM_STATE)
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.scatter(X[:,0],X[:,1],s=14)
ax.set_xlabel("anomalia e temperaturës"); ax.set_ylabel("anomalia e presionit")
ax.set_title("Të dhëna pa etiketa")
plt.show()

**Interpretim.** Figura duhet lexuar si test i një hipoteze të vogël: a përputhet forma e vëzhguar me atë që parashikuam nga matematika?


# 2. Funksioni objektiv i k-means — CORE

k-means minimizon
$$
J=\sum_{k=1}^{K}\sum_{x_i\in C_k}\|x_i-\mu_k\|^2.
$$
Algoritmi alternon ndërmjet caktimit te qendra më e afërt dhe rillogaritjes së qendrave.

# 3. Iteracionet manuale — CORE

### Para se të ekzekutojmë: k-means hap pas hapi

Ruajmë qendrat pas çdo iteracioni që të shohim procesin, jo vetëm rezultatin final. Caktimi dhe përditësimi alternohen derisa lëvizja e qendrave të bëhet e papërfillshme.

**Parashikim para ekzekutimit.** Vendosni çfarë forme duhet të ketë figura nëse argumenti teorik është i saktë. Grafiku përdoret për të testuar një pritshmëri, jo vetëm për ta ilustruar.


In [ ]:
# Qëllimi i këtij blloku: k-means hap pas hapi.
K=3
centers=X[rng.choice(len(X),K,replace=False)].copy()
history=[centers.copy()]
labels_history=[]
# Përsërisim procedurën për të studiuar konvergjencën ose ndjeshmërinë ndaj parametrave.
for _ in range(8):
    d2=((X[:,None,:]-centers[None,:,:])**2).sum(axis=2)
    labels=d2.argmin(axis=1)
    labels_history.append(labels.copy())
    new=np.array([X[labels==k].mean(axis=0) for k in range(K)])
    history.append(new.copy())
    if np.linalg.norm(new-centers)<1e-6: break
    centers=new

for step in [0,1,min(2,len(labels_history)-1),len(labels_history)-1]:
    # Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
    fig,ax=plt.subplots()
    lab=labels_history[min(step,len(labels_history)-1)]
    ax.scatter(X[:,0],X[:,1],c=lab,s=10)
    c=history[step]
    ax.scatter(c[:,0],c[:,1],s=130,marker="x")
    ax.set_title(f"k-means: iteracioni {step}")
    plt.show()

**Interpretim.** Figura duhet lexuar si test i një hipoteze të vogël: a përputhet forma e vëzhguar me atë që parashikuam nga matematika?


### Zbatimi kanonik i k-means në `src/`

### Para se të ekzekutojmë: Kontroll me implementimin kanonik

Përdorim modulin `KMeansFromScratch` për të parë inertia-n dhe numrin e iteracioneve. Kjo lidh demonstrimin vizual me një implementim që mund të përdoret në notebook-e të tjera.

**Parashikim para ekzekutimit.** Mendoni ku do të zhvendosen qendrat pas hapit të parë. Algoritmi duhet të ulë funksionin objektiv, por jo domosdoshmërisht të gjejë të njëjtin minimum për çdo inicializim.


In [ ]:
# Qëllimi i këtij blloku: Kontroll me implementimin kanonik.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import sys
# Kërkojmë automatikisht rrënjën e repository-t, në mënyrë që notebook-u të funksionojë nga dosje të ndryshme.
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir():
        sys.path.insert(0, str(candidate))
        break
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from src.clustering.kmeans import KMeansFromScratch
# Përshtatim parametrat e modelit mbi bashkësinë e trajnimit.
km_src = KMeansFromScratch(n_clusters=3, seed=RANDOM_STATE).fit(X)
print("Inertia nga src:", round(km_src.inertia_, 3), "iteracione:", km_src.n_iter_)

**Interpretim.** Konvergjenca e k-means garanton stabilizim të procedurës për atë inicializim, jo zbulimin e një “të vërtete” unike për strukturën e të dhënave.


# 4. Elbow dhe silhouette — CORE

### Para se të ekzekutojmë: Sa grupe duhen?

Elbow dhe silhouette japin informacione të ndryshme. Elbow kërkon një ndryshim të ritmit të uljes së inertia-s; silhouette krahason kohezionin brenda grupit me ndarjen nga grupet e tjera. Asnjëra nuk zëvendëson interpretimin e problemit.

**Parashikim para ekzekutimit.** Mendoni ku do të zhvendosen qendrat pas hapit të parë. Algoritmi duhet të ulë funksionin objektiv, por jo domosdoshmërisht të gjejë të njëjtin minimum për çdo inicializim.


In [ ]:
# Qëllimi i këtij blloku: Sa grupe duhen?.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.cluster import KMeans
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.metrics import silhouette_score
ks=range(2,8); inertia=[]; sil=[]
for k in ks:
    # Përshtatim parametrat e modelit mbi bashkësinë e trajnimit.
    m=KMeans(k,random_state=RANDOM_STATE,n_init=10).fit(X)
    inertia.append(m.inertia_)
    sil.append(silhouette_score(X,m.labels_))

# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.plot(list(ks),inertia,marker="o")
ax.set_xlabel("k"); ax.set_ylabel("Inertia")
ax.set_title("Metoda e bërrylit")
plt.show()

# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.plot(list(ks),sil,marker="o")
ax.set_xlabel("k"); ax.set_ylabel("Silhouette")
ax.set_title("Silhouette score")
plt.show()

**Interpretim.** Konvergjenca e k-means garanton stabilizim të procedurës për atë inicializim, jo zbulimin e një “të vërtete” unike për strukturën e të dhënave.


# 5. Grupimi hierarkik — CORE

### Para se të ekzekutojmë: Hierarkia e ngjashmërisë

Dendrogrami nuk kërkon të fiksojmë k që në fillim. Ai tregon rendin dhe distancat e bashkimit të grupeve, duke dhënë një pamje shumë-shkallëshe të strukturës.

**Parashikim para ekzekutimit.** Lidhjet në lartësi të vogël përfaqësojnë objekte/grupe më të afërta. Një prerje horizontale e dendrogramit përcakton numrin e grupeve.


In [ ]:
# Qëllimi i këtij blloku: Hierarkia e ngjashmërisë.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from scipy.cluster.hierarchy import linkage,dendrogram
Z=linkage(X[:80],method="ward")
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots(figsize=(10,5))
dendrogram(Z,no_labels=True,ax=ax)
ax.set_ylabel("Distanca e bashkimit")
ax.set_title("Dendrogram")
plt.show()

**Interpretim.** Dendrogrami ruan informacion mbi shkallën e bashkimit; numri i grupeve del nga zgjedhja e një niveli prerjeje dhe duhet justifikuar nga konteksti.


---
## Deri këtu: materiali bazë për orën mësimore

Në një seancë rreth 60 minuta, zakonisht ndalemi këtu. Seksionet që vijojnë janë për thellim,
ushtrime të pavarura dhe eksperimentim.

# 6. Kur k-means dështon — EXTENDED

### Para se të ekzekutojmë: Një dështim i qëllimshëm i k-means

Dy gjysmëhëna kanë strukturë të qartë për syrin, por qendrat e k-means favorizojnë grupe afërsisht sferike. Ky shembull na kujton se çdo algoritëm grupimi përmban një paragjykim gjeometrik.

**Parashikim para ekzekutimit.** Mendoni ku do të zhvendosen qendrat pas hapit të parë. Algoritmi duhet të ulë funksionin objektiv, por jo domosdoshmërisht të gjejë të njëjtin minimum për çdo inicializim.


In [ ]:
# Qëllimi i këtij blloku: Një dështim i qëllimshëm i k-means.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.datasets import make_moons
# Gjenerojmë të dhëna të kontrolluara që na lejojnë të dimë strukturën që duhet të zbulojë modeli.
Xm,_=make_moons(n_samples=450,noise=0.07,random_state=RANDOM_STATE)
# Përshtatim parametrat e modelit mbi bashkësinë e trajnimit.
km=KMeans(2,random_state=RANDOM_STATE,n_init=10).fit(Xm)
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.scatter(Xm[:,0],Xm[:,1],c=km.labels_,s=12)
ax.set_title("k-means imponon grupe afërsisht sferike")
plt.show()

**Interpretim.** Konvergjenca e k-means garanton stabilizim të procedurës për atë inicializim, jo zbulimin e një “të vërtete” unike për strukturën e të dhënave.


**Interpretim.** k-means optimizon një funksion të caktuar gjeometrik; ai nuk 'zbulon' domosdoshmërisht grupet që ne konsiderojmë fizikisht të kuptimshme.

Në një analizë reale ky interpretim duhet të shoqërohet me kontroll të njësive, madhësive karakteristike dhe hipotezave që lejojnë kalimin nga një tregues numerik te një përfundim fizik.

# 7. Aplikim jo-fizik: segmentim klientësh — EXTENDED

Në marketing, të njëjtat mjete përdoren për segmentim, por interpretimi i grupeve duhet të bëhet me kujdes: grupet algoritmike nuk janë kategori natyrore të garantuara.

## Punishtja — ndrysho, mat dhe shpjego

Në këtë pjesë nuk kërkohet ekzekutim pasiv. Për çdo eksperiment:

1. shkruani paraprakisht çfarë prisni të ndodhë;
2. ndryshoni **vetëm një** element në të njëjtën kohë;
3. ruani metrikën ose figurën që ndryshon;
4. shpjegoni mekanizmin matematik ose statistikor;
5. dalloni qartë atë që tregon eksperimenti nga ajo që **nuk** mund të përfundohet prej tij.

### Pyetje për diskutim

1. Pse k-means favorizon grupe afërsisht sferike?
2. Çfarë mat silhouette score?
3. Pse elbow method nuk jep gjithmonë një përgjigje të vetme?
4. Çfarë informacioni shton dendrogrami?


# Ushtrime — PLAYGROUND

        ## A. Ushtrime të shkurtra
        1. Përsëritni k-means me 20 seed dhe analizoni ndryshueshmërinë.
2. Krahasoni k=2,3,4 me silhouette.
3. Ndryshoni `linkage` në dendrogram.
4. Shpjegoni pse scaling ndryshon grupimin.

        ## B. Ushtrim i plotë
        Gjeneroni tri faza fizike me forma jo-sferike dhe krahasoni k-means me AgglomerativeClustering.

        ## C. Playground
        Ndryshoni k, seed, zhurmën dhe standardizimin.

In [ ]:
# TODO / PLAYGROUND
# Ndryshoni parametrat, shtoni grafikë ose krahasoni modele të tjera.

## Përmbledhje konceptuale e kapitullit

Notebook-u **Grupimi dhe zbulimi i strukturës** duhet të lihet me një ide të qartë: algoritmi është vetëm një hallkë e arsyetimit. Para tij kemi zgjedhjen e përfaqësimit dhe hipotezat mbi të dhënat; pas tij kemi validimin, diagnostikën dhe interpretimin. Nëse një student mund të ekzekutojë të gjitha qelizat por nuk mund të shpjegojë pse secila prej tyre është e nevojshme, atëherë puna llogaritëse nuk është përfunduar ende.

Për studim të mëtejshëm, rikthehuni te shënimet e leksionit për derivimet e plota dhe përdoreni këtë notebook si laborator ku këto ide testohen numerikisht.

## Kontrolli përfundimtar

- [ ] Shpjegoj hapat assignment/update.
- [ ] Interpretoj inertia.
- [ ] Përdor silhouette me kujdes.
- [ ] Lexoj një dendrogram.
- [ ] Tregoj një rast ku k-means jep ndarje mashtruese.
